**Voice-to-Grasp - Experiment 3**: predictions of the base and the kitchen-specialised model on the
kitchen and repair test sets (paper Sections 2.6 and 3.3). Outputs were cleared for the release;
the paper's numbers come from `Revision_ISR_2026/10_results_numbers.py`.

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
id2label = {
    0: "grasp",
    1: "open_hand",
    2: "gentle_grasp",
    3: "tight_grasp",
    4: "pinch_grasp",
    5: "tighten",
    6: "loosen"
}

In [ ]:
model_path = r"D:\PythonEnvs\nlpglove\bert_finetuned_updated"
# model_path = r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments\Tester-21\adaptive_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()

print("Model Loaded Successfully")

In [ ]:
def predict_intent(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item()

    return id2label[predicted_class]

In [ ]:
# file_path = r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments\UserStudy_code_analysis\VoiceToGrasp_StudySheet_PilotA.xlsx"  # Change this
file_path = r"D:\PythonEnvs\nlpglove\Scripts_ContinualLearning\kitchen_test_nonnative.xlsx"
df = pd.read_excel(file_path)
# df = pd.read_csv(file_path)

df.head()

In [ ]:
predictions = []

for prompt in df["prompt"]:
    
    pred = predict_intent(prompt)
    
    predictions.append(pred)

df["predicted_grasp"] = predictions

df.head()

In [ ]:
output_path = r"D:\PythonEnvs\nlpglove\Scripts_ContinualLearning\Result_csv\kitchen_predictions_adaptive.csv"


print("Predictions saved to:", output_path)

In [ ]:
df.to_csv(output_path, index=False)

accuracy = accuracy_score(df["true_grasp"], df["predicted_grasp"])

print("Accuracy:", accuracy)

In [ ]:
print(classification_report(
    df["true_grasp"],
    df["predicted_grasp"]
))

In [ ]:
# print the confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(df["true_grasp"], df["predicted_grasp"], labels=list(id2label.values()))
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=list(id2label.values()), yticklabels=list(id2label.values()))
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')